# Linked Lists, Stacks, Queues and Monotonic Structures

Notebook for **Chapter 5** of the *dsa* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/dsa/ch05.html

Tip: change a number, predict the output, then run the cell again.

## 5.1 Linked Lists From Scratch

In [ ]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val      # data stored in this box
        self.next = next    # next box, or None at the end

def build(values):
    head = None
    for v in reversed(values):      # build from the back: no tail variable needed
        head = ListNode(v, head)
    return head

def to_list(head):
    out, cur = [], head
    while cur:                      # the walk loop: stop when cur is None
        out.append(cur.val)
        cur = cur.next
    return out

head = build([3, 7, 9])
node = head.next                    # the 7-box
new = ListNode(8)
new.next = node.next                # 1. point the new node at the rest
node.next = new                     # 2. then connect the front
print(to_list(head))

[3, 7, 8, 9]


In [ ]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

class LinkedList:
    def __init__(self):
        self.head = None

    def push_front(self, v):
        self.head = ListNode(v, self.head)

    def append(self, v):
        if self.head is None:
            self.head = ListNode(v)
            return
        cur = self.head
        while cur.next:                # walk to the last node
            cur = cur.next
        cur.next = ListNode(v)

    def find(self, v):
        cur = self.head
        while cur and cur.val != v:
            cur = cur.next
        return cur is not None

    def delete(self, v):
        if self.head and self.head.val == v:   # the head has no node before it
            self.head = self.head.next
            return
        cur = self.head
        while cur and cur.next and cur.next.val != v:
            cur = cur.next                     # stop one node BEFORE the target
        if cur and cur.next:
            cur.next = cur.next.next

    def to_list(self):
        out, cur = [], self.head
        while cur:
            out.append(cur.val)
            cur = cur.next
        return out

ll = LinkedList()
ll.push_front(3); ll.append(7); ll.append(9)
assert ll.to_list() == [3, 7, 9] and ll.find(7) and not ll.find(5)
ll.delete(7); assert ll.to_list() == [3, 9]
ll.delete(3); ll.delete(42); assert ll.to_list() == [9]
ll.delete(9); assert ll.to_list() == []
print("linked list ok")

linked list ok


In [ ]:
class DNode:
    def __init__(self, val):
        self.val, self.prev, self.next = val, None, None

class DList:
    def __init__(self):
        self.head = self.tail = None

    def append(self, v):
        node = DNode(v)
        if self.tail is None:
            self.head = self.tail = node
        else:
            node.prev, self.tail.next = self.tail, node
            self.tail = node
        return node

    def remove(self, node):
        if node.prev: node.prev.next = node.next
        else: self.head = node.next            # node was the head
        if node.next: node.next.prev = node.prev
        else: self.tail = node.prev            # node was the tail

    def forward(self):
        out, cur = [], self.head
        while cur:
            out.append(cur.val); cur = cur.next
        return out

d = DList()
a, b, c = d.append(1), d.append(2), d.append(3)
d.remove(b); assert d.forward() == [1, 3]
d.remove(a); assert d.head is c and d.tail is c
d.remove(c); assert d.head is None and d.tail is None
print("doubly linked list ok")

doubly linked list ok


## 5.2 Reversing a Linked List

In [ ]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def build(a):
    head = None
    for v in reversed(a):
        head = ListNode(v, head)
    return head

def to_list(h):
    out = []
    while h:
        out.append(h.val)
        h = h.next
    return out

def reverse(head):
    prev, cur = None, head      # the reversed part starts empty
    while cur:
        nxt = cur.next          # 1. save the rest of the list
        cur.next = prev         # 2. turn the arrow around
        prev = cur              # 3. grow the reversed part
        cur = nxt               # 4. move on to the saved rest
    return prev                 # prev is the new head

assert to_list(reverse(build([]))) == []
assert to_list(reverse(build([5]))) == [5]
print(to_list(reverse(build([1, 2, 3, 4]))))

[4, 3, 2, 1]


In [ ]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def build(a):
    head = None
    for v in reversed(a):
        head = ListNode(v, head)
    return head

def to_list(h):
    out = []
    while h:
        out.append(h.val)
        h = h.next
    return out

def reverse_k_group(head, k):
    dummy = ListNode(0, head)
    group_prev = dummy
    while True:
        kth = group_prev
        for _ in range(k):             # count first: do k nodes exist?
            kth = kth.next
            if kth is None:
                return dummy.next
        prev, cur = kth.next, group_prev.next   # first node will point past the group
        for _ in range(k):
            nxt = cur.next
            cur.next = prev
            prev, cur = cur, nxt
        old_first = group_prev.next    # now the group's last node
        group_prev.next = prev         # prev is the group's new first node
        group_prev = old_first

assert to_list(reverse_k_group(build([1, 2, 3, 4, 5]), 3)) == [3, 2, 1, 4, 5]
assert to_list(reverse_k_group(build([1, 2, 3]), 1)) == [1, 2, 3]
assert to_list(reverse_k_group(build([1, 2]), 5)) == [1, 2]
print(to_list(reverse_k_group(build([1, 2, 3, 4, 5]), 2)))

[2, 1, 4, 3, 5]


## 5.3 Fast and Slow Pointers: Middle and Cycle

In [ ]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def has_cycle(head):
    slow = fast = head
    while fast and fast.next:        # fast needs two nodes ahead of it
        slow = slow.next             # 1 step
        fast = fast.next.next        # 2 steps
        if slow is fast:             # same object, not same value
            return True
    return False

def middle(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    return slow                      # for even length: the second middle

def make(n):                         # chain of n nodes holding 1..n
    nodes = [ListNode(i + 1) for i in range(n)]
    for a, b in zip(nodes, nodes[1:]):
        a.next = b
    return nodes

for n in (1, 2, 5, 6):
    print(n, "nodes: middle is", middle(make(n)[0]).val)
chain = make(8)
assert not has_cycle(chain[0])
chain[7].next = chain[2]             # last node points back to node 3
assert has_cycle(chain[0]) and not has_cycle(None)

1 nodes: middle is 1
2 nodes: middle is 2
5 nodes: middle is 3
6 nodes: middle is 4


In [ ]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def cycle_entry(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
        if slow is fast:                 # phase 1: met inside the cycle
            finder = head
            while finder is not slow:    # phase 2: one step each
                finder = finder.next
                slow = slow.next
            return finder
    return None

def link(n, back_to):                    # nodes 0..n-1; last points to back_to (or None)
    nodes = [ListNode(i) for i in range(n)]
    for a, b in zip(nodes, nodes[1:]):
        a.next = b
    if back_to is not None:
        nodes[-1].next = nodes[back_to]
    return nodes

assert cycle_entry(link(8, 3)[0]).val == 3
assert cycle_entry(link(4, 0)[0]).val == 0     # whole list is the cycle
assert cycle_entry(link(1, 0)[0]).val == 0     # node points at itself
assert cycle_entry(link(4, None)[0]) is None and cycle_entry(None) is None
print("entry:", cycle_entry(link(8, 3)[0]).val)

entry: 3


In [ ]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def build(a):
    head = None
    for v in reversed(a):
        head = ListNode(v, head)
    return head

def is_palindrome(head):
    slow = fast = head
    while fast and fast.next:          # 1. find the middle
        slow, fast = slow.next, fast.next.next
    prev, cur = None, slow             # 2. reverse the second half
    while cur:
        cur.next, prev, cur = prev, cur, cur.next
    left, right = head, prev           # 3. compare the halves
    while right:
        if left.val != right.val:
            return False
        left, right = left.next, right.next
    return True

assert is_palindrome(build([1, 2, 3, 2, 1])) and is_palindrome(build([1])) and is_palindrome(None)
assert not is_palindrome(build([1, 2, 3]))
print(is_palindrome(build([1, 2, 2, 1])), is_palindrome(build([1, 2, 3])))

True False


## 5.4 The Dummy-Node Trick: Merge and Remove

In [ ]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def build(a):
    head = None
    for v in reversed(a):
        head = ListNode(v, head)
    return head

def to_list(h):
    out = []
    while h:
        out.append(h.val)
        h = h.next
    return out

def merge(a, b):
    dummy = ListNode()              # fake first node
    tail = dummy                    # last node of the output so far
    while a and b:
        if a.val <= b.val:
            tail.next, a = a, a.next
        else:
            tail.next, b = b, b.next
        tail = tail.next
    tail.next = a if a else b       # attach the leftover list in one move
    return dummy.next               # skip the dummy

assert to_list(merge(build([]), build([5]))) == [5]
assert to_list(merge(build([]), build([]))) == []
assert to_list(merge(build([1, 1]), build([1]))) == [1, 1, 1]
print(to_list(merge(build([1, 4, 6]), build([2, 3, 7]))))

[1, 2, 3, 4, 6, 7]


In [ ]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def build(a):
    head = None
    for v in reversed(a):
        head = ListNode(v, head)
    return head

def to_list(h):
    out = []
    while h:
        out.append(h.val)
        h = h.next
    return out

def remove_nth_from_end(head, n):
    dummy = ListNode(0, head)
    slow = fast = dummy
    for _ in range(n):               # open a gap of n nodes
        fast = fast.next
    while fast.next:                 # run until fast is on the last node
        slow, fast = slow.next, fast.next
    slow.next = slow.next.next       # slow is just before the target
    return dummy.next

assert to_list(remove_nth_from_end(build([1, 2, 3, 4, 5]), 5)) == [2, 3, 4, 5]
assert to_list(remove_nth_from_end(build([1, 2, 3, 4, 5]), 1)) == [1, 2, 3, 4]
assert to_list(remove_nth_from_end(build([7]), 1)) == []
print(to_list(remove_nth_from_end(build([1, 2, 3, 4, 5]), 2)))

[1, 2, 3, 5]


In [ ]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def build(a):
    head = None
    for v in reversed(a):
        head = ListNode(v, head)
    return head

def to_list(h):
    out = []
    while h:
        out.append(h.val)
        h = h.next
    return out

def merge(a, b):
    dummy = tail = ListNode()
    while a and b:
        if a.val <= b.val:
            tail.next, a = a, a.next
        else:
            tail.next, b = b, b.next
        tail = tail.next
    tail.next = a if a else b
    return dummy.next

def sort_list(head):
    if head is None or head.next is None:      # 0 or 1 node: sorted
        return head
    slow, fast = head, head.next               # fast one ahead: slow ends on the LEFT middle
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
    second, slow.next = slow.next, None        # cut the list in two
    return merge(sort_list(head), sort_list(second))

assert to_list(sort_list(build([5, 1, 4, 1, 3, 9, 0]))) == [0, 1, 1, 3, 4, 5, 9]
assert to_list(sort_list(build([]))) == [] and to_list(sort_list(build([2, 1]))) == [1, 2]
print(to_list(sort_list(build([4, 2, 1, 3]))))

[1, 2, 3, 4]


## 5.5 Stacks: Last In, First Out

In [ ]:
def is_valid(s):
    pairs = {")": "(", "]": "[", "}": "{"}    # closer -> its opener
    stack = []
    for ch in s:
        if ch in pairs.values():
            stack.append(ch)                   # opener: remember it
        elif not stack or stack.pop() != pairs[ch]:
            return False                       # nothing to match, or wrong type
    return not stack                           # leftover openers mean invalid

tests = {"": True, "()[]{}": True, "{[()]}": True, "(]": False,
         "([)]": False, ")(": False, "((": False, "]": False}
for text, expected in tests.items():
    assert is_valid(text) == expected, text
print(is_valid("{[()]}"), is_valid("([)]"), is_valid("(("))

True False False


In [ ]:
class MinStack:
    def __init__(self):
        self.items = []            # pairs (value, minimum so far)

    def push(self, x):
        low = min(x, self.items[-1][1]) if self.items else x
        self.items.append((x, low))

    def pop(self):
        return self.items.pop()[0]

    def top(self):
        return self.items[-1][0]

    def get_min(self):
        return self.items[-1][1]

s = MinStack()
mins = []
for x in [5, 3, 7, 3, 8]:
    s.push(x)
    mins.append(s.get_min())
after = []
for _ in range(4):
    s.pop()
    after.append(s.get_min())
print(mins, after)
assert mins == [5, 3, 3, 3, 3] and after == [3, 3, 3, 5] and s.top() == 5

[5, 3, 3, 3, 3] [3, 3, 3, 5]


In [ ]:
def calculate(s):
    stack, num, op = [], 0, "+"     # op = operator that came BEFORE num
    for ch in s + "+":              # the extra "+" flushes the last number
        if ch.isdigit():
            num = num * 10 + int(ch)
        elif ch != " ":
            if op == "+": stack.append(num)
            elif op == "-": stack.append(-num)
            elif op == "*": stack.append(stack.pop() * num)
            else: stack.append(int(stack.pop() / num))
            op, num = ch, 0
    return sum(stack)

tests = {"3+2*2": 7, " 3/2 ": 1, " 3+5 / 2 ": 5, "14-3*2+8/4": 10,
         "2-3*4": -10, "10/3*3": 9, "0-5/2": -2, "42": 42}
for text, expected in tests.items():
    assert calculate(text) == expected, text
print(calculate("14-3*2+8/4"), calculate("0-5/2"))

10 -2


## 5.6 Queues and Deques: First In, First Out

In [ ]:
from collections import deque

q = deque()
q.append("A")          # enqueue at the back
q.append("B")
q.append("C")
front = q.popleft()    # dequeue from the front, O(1)
print(front, q)
q.appendleft("Z")      # a deque can also add at the front
last = q.pop()         # and remove at the back
print(last, q)
print(len(q), q[0], q[-1])

A deque(['B', 'C'])
C deque(['Z', 'B'])
2 Z B


In [ ]:
from collections import deque

class RecentCounter:
    def __init__(self):
        self.times = deque()

    def ping(self, t):
        self.times.append(t)
        while self.times[0] < t - 3000:    # strictly older than the window
            self.times.popleft()
        return len(self.times)

rc = RecentCounter()
results = [rc.ping(t) for t in (1, 100, 3001, 3002)]
print(results)
assert results == [1, 2, 3, 3]

[1, 2, 3, 3]


## 5.7 The Monotonic Stack

In [ ]:
def next_greater_indexes(nums):
    answer = [-1] * len(nums)          # -1 means "no greater element to the right"
    stack = []                         # indexes still waiting; their values decrease
    for i, x in enumerate(nums):
        while stack and nums[stack[-1]] < x:   # x beats the waiting item
            answer[stack.pop()] = i            # the popped item found its answer
        stack.append(i)                        # i now waits for its own answer
    return answer

print(next_greater_indexes([73, 74, 75, 71, 69, 72, 76, 73]))

[1, 2, 6, 5, 5, 6, -1, -1]


In [ ]:
def next_greater(nums):
    answer = [-1] * len(nums)
    stack = []
    for i, x in enumerate(nums):
        while stack and nums[stack[-1]] < x:
            answer[stack.pop()] = x
        stack.append(i)
    return answer

assert next_greater([]) == [] and next_greater([3, 3, 3]) == [-1, -1, -1]
assert next_greater([5, 4, 3]) == [-1, -1, -1]
print(next_greater([2, 1, 2, 4, 3]))

[4, 2, 4, -1, -1]


In [ ]:
def daily_temperatures(temps):
    wait = [0] * len(temps)
    stack = []                         # days still waiting for a warmer day
    for i, t in enumerate(temps):
        while stack and temps[stack[-1]] < t:
            j = stack.pop()
            wait[j] = i - j            # days waited
        stack.append(i)
    return wait

result = daily_temperatures([73, 74, 75, 71, 69, 72, 76, 73])
print(result)
assert result == [1, 1, 4, 2, 1, 1, 0, 0]
assert daily_temperatures([30, 40, 50, 60]) == [1, 1, 1, 0]
assert daily_temperatures([50, 50]) == [0, 0] and daily_temperatures([]) == []

[1, 1, 4, 2, 1, 1, 0, 0]


In [ ]:
import random

def largest_rectangle(heights):
    stack = []                                 # indexes, heights increasing
    best = 0
    for i, h in enumerate(heights + [0]):      # the 0 sentinel flushes the stack
        while stack and heights[stack[-1]] >= h:
            height = heights[stack.pop()]
            left = stack[-1] if stack else -1  # previous smaller bar, or the left wall
            best = max(best, height * (i - left - 1))
        stack.append(i)
    return best

def brute(heights):
    best = 0
    for left in range(len(heights)):
        lowest = heights[left]
        for right in range(left, len(heights)):
            lowest = min(lowest, heights[right])
            best = max(best, lowest * (right - left + 1))
    return best

assert largest_rectangle([]) == 0 and largest_rectangle([5]) == 5
assert largest_rectangle([3, 3, 3, 3]) == 12 and largest_rectangle([1, 2, 3, 4, 5]) == 9
assert largest_rectangle([5, 0, 5]) == 5
random.seed(11)
for _ in range(300):
    data = [random.randint(0, 8) for _ in range(random.randint(0, 10))]
    assert largest_rectangle(data) == brute(data)
print(largest_rectangle([2, 1, 5, 6, 2, 3]))

10


## 5.8 The Monotonic Deque: Sliding Window Maximum

In [ ]:
from collections import deque
import random

def window_max(nums, k):
    dq = deque()                           # indexes; values decrease front to back
    out = []
    for i, x in enumerate(nums):
        while dq and nums[dq[-1]] <= x:    # 1. drop weaker items from the back
            dq.pop()
        dq.append(i)                       # 2. add the new index
        if dq[0] <= i - k:                 # 3. drop the front if it left the window
            dq.popleft()
        if i >= k - 1:                     # 4. window is full: the front is the maximum
            out.append(nums[dq[0]])
    return out

def brute(nums, k):
    return [max(nums[i:i + k]) for i in range(len(nums) - k + 1)]

assert window_max([4, 3, 2, 1], 2) == [4, 3, 2] and window_max([2, 2, 2], 2) == [2, 2]
assert window_max([9, 8, 7], 1) == [9, 8, 7] and window_max([5, 1, 4], 3) == [5]
random.seed(3)
for _ in range(300):
    n = random.randint(1, 15)
    data = [random.randint(-5, 5) for _ in range(n)]
    k = random.randint(1, n)
    assert window_max(data, k) == brute(data, k)
print(window_max([1, 3, -1, -3, 5, 3, 6, 7], 3))

[3, 3, 5, 5, 6, 7]


In [ ]:
from collections import deque

def longest_subarray(nums, limit):
    big, small = deque(), deque()    # indexes: values decreasing / increasing
    left = best = 0
    for right, x in enumerate(nums):
        while big and nums[big[-1]] <= x:
            big.pop()
        big.append(right)
        while small and nums[small[-1]] >= x:
            small.pop()
        small.append(right)
        while nums[big[0]] - nums[small[0]] > limit:   # too uneven: shrink from the left
            left += 1
            if big[0] < left:
                big.popleft()
            if small[0] < left:
                small.popleft()
        best = max(best, right - left + 1)
    return best

assert longest_subarray([10, 1, 2, 4, 7, 2], 5) == 4
assert longest_subarray([1, 5, 6, 7, 8, 10, 6, 5, 6], 4) == 5
print(longest_subarray([10, 1, 2, 4, 7, 2], 5))

4


## Chapter recap

## Practice set

Try each problem before opening the answer. Code in problem statements is run for you in the answer.

In [ ]:
s = []
s.append(1)
s.append(2)
s.append(3)
s.pop()
s.append(4)
print(s, s[-1])

In [ ]:
from collections import deque
d = deque([1, 2, 3])
d.appendleft(0)
d.rotate(2)
d.popleft()
print(d)

In [ ]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def reverse(head):
    prev, cur = None, head
    while cur:
        cur.next = prev
        prev = cur
        cur = cur.next
    return prev

head = ListNode(1, ListNode(2, ListNode(3)))
node = reverse(head)
out = []
while node:
    out.append(node.val)
    node = node.next
print(out)

In [ ]:
def is_valid(s):
    pairs = {")": "("}
    stack = []
    for ch in s:
        if ch == "(":
            stack.append(ch)
        elif not stack or stack.pop() != pairs[ch]:
            return False
    return True

print(is_valid("(("))

In [ ]:
def min_add(s):
    need = bal = 0
    for ch in s:
        if ch == "(":
            bal += 1
        elif bal:
            bal -= 1            # this closer matches an earlier opener
        else:
            need += 1           # a closer with nothing to match: one "(" must be added
    return need + bal           # plus one ")" for each unmatched opener

assert min_add("())") == 1 and min_add("(((") == 3 and min_add("") == 0
print(min_add("()))(("))

4


In [ ]:
def eval_rpn(tokens):
    stack = []
    for t in tokens:
        if t in ("+", "-", "*", "/"):
            b = stack.pop()
            a = stack.pop()
            stack.append({"+": a + b, "-": a - b, "*": a * b,
                          "/": int(a / b) if t == "/" else 0}[t])
        else:
            stack.append(int(t))
    return stack[0]

assert eval_rpn(["5", "2", "-"]) == 3 and eval_rpn(["7", "-2", "/"]) == -3
print(eval_rpn(["4", "13", "5", "/", "+"]), eval_rpn(["2", "1", "+", "3", "*"]))

6 9


In [ ]:
class QueueFromStacks:
    def __init__(self):
        self.inbox, self.outbox = [], []

    def push(self, x):
        self.inbox.append(x)

    def pop(self):
        if not self.outbox:
            while self.inbox:
                self.outbox.append(self.inbox.pop())
        return self.outbox.pop()

q = QueueFromStacks()
for x in (1, 2, 3):
    q.push(x)
first = q.pop()
q.push(4)
print(first, q.pop(), q.pop(), q.pop())

1 2 3 4


In [ ]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def build(a):
    head = None
    for v in reversed(a):
        head = ListNode(v, head)
    return head

def to_list(h):
    out = []
    while h:
        out.append(h.val)
        h = h.next
    return out

def reverse_between(head, left, right):
    dummy = ListNode(0, head)
    before = dummy
    for _ in range(left - 1):
        before = before.next
    first = before.next
    for _ in range(right - left):
        moving = first.next
        first.next = moving.next
        moving.next = before.next
        before.next = moving
    return dummy.next

assert to_list(reverse_between(build([1, 2, 3]), 1, 3)) == [3, 2, 1]
print(to_list(reverse_between(build([1, 2, 3, 4, 5]), 2, 4)))

[1, 4, 3, 2, 5]


In [ ]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def build(a):
    head = None
    for v in reversed(a):
        head = ListNode(v, head)
    return head

def to_list(h):
    out = []
    while h:
        out.append(h.val)
        h = h.next
    return out

def delete_middle(head):
    dummy = ListNode(0, head)
    slow, fast = dummy, head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
    slow.next = slow.next.next
    return dummy.next

assert to_list(delete_middle(build([7]))) == []
assert to_list(delete_middle(build([1, 2, 3, 4, 5]))) == [1, 2, 4, 5]
print(to_list(delete_middle(build([1, 2, 3, 4]))))

[1, 2, 4]


In [ ]:
def wait_days(temps):
    wait = [0] * len(temps)
    stack = []
    for i, t in enumerate(temps):
        while stack and temps[stack[-1]] <= t:
            j = stack.pop()
            wait[j] = i - j
        stack.append(i)
    return wait

print(wait_days([3, 3, 5]))

In [ ]:
def stock_span(prices):
    spans, stack = [], []
    for i, p in enumerate(prices):
        while stack and prices[stack[-1]] <= p:
            stack.pop()
        spans.append(i - stack[-1] if stack else i + 1)
        stack.append(i)
    return spans

assert stock_span([]) == [] and stock_span([5, 4, 3]) == [1, 1, 1]
print(stock_span([100, 80, 60, 70, 60, 75, 85]))

[1, 1, 1, 2, 1, 4, 6]


In [ ]:
def next_greater_circular(nums):
    n = len(nums)
    answer, stack = [-1] * n, []
    for i in range(2 * n):
        x = nums[i % n]
        while stack and nums[stack[-1]] < x:
            answer[stack.pop()] = x
        if i < n:
            stack.append(i)
    return answer

assert next_greater_circular([5, 4, 3, 2, 1]) == [-1, 5, 5, 5, 5]
assert next_greater_circular([2, 2, 2]) == [-1, -1, -1]
print(next_greater_circular([1, 2, 1]))

[2, -1, 2]


In [ ]:
def remove_k_digits(num, k):
    stack = []
    for d in num:
        while k and stack and stack[-1] > d:
            stack.pop()
            k -= 1
        stack.append(d)
    if k:
        stack = stack[:-k]
    return "".join(stack).lstrip("0") or "0"

assert remove_k_digits("10", 2) == "0" and remove_k_digits("12345", 2) == "123"
assert remove_k_digits("10200", 1) == "200"
print(remove_k_digits("1432219", 3))

1219


In [ ]:
def decode(s):
    stack, cur, num = [], "", 0
    for ch in s:
        if ch.isdigit():
            num = num * 10 + int(ch)
        elif ch == "[":
            stack.append((cur, num))
            cur, num = "", 0
        elif ch == "]":
            prev, k = stack.pop()
            cur = prev + cur * k
        else:
            cur += ch
    return cur

assert decode("2[ab]3[c]") == "ababccc" and decode("abc") == "abc"
assert decode("10[x]") == "x" * 10
print(decode("3[a2[c]]"))

accaccacc


In [ ]:
def asteroid_collision(asteroids):
    stack = []
    for a in asteroids:
        alive = True
        while alive and a < 0 and stack and stack[-1] > 0:
            if stack[-1] < -a:
                stack.pop()              # the right-mover is smaller: it explodes
            else:
                if stack[-1] == -a:
                    stack.pop()          # both explode
                alive = False
        if alive:
            stack.append(a)
    return stack

assert asteroid_collision([5, 10, -5]) == [5, 10]
assert asteroid_collision([8, -8]) == [] and asteroid_collision([-2, -1, 1, 2]) == [-2, -1, 1, 2]
print(asteroid_collision([10, 2, -5]))

[10]


In [ ]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def build(a):
    head = None
    for v in reversed(a):
        head = ListNode(v, head)
    return head

def to_list(h):
    out = []
    while h:
        out.append(h.val)
        h = h.next
    return out

def reorder(head):
    if not head or not head.next:
        return head
    slow, fast = head, head.next
    while fast and fast.next:                  # slow ends on the left middle
        slow, fast = slow.next, fast.next.next
    second, slow.next = slow.next, None        # cut
    prev = None
    while second:                              # reverse the second half
        second.next, prev, second = prev, second, second.next
    first, second = head, prev
    while second:                              # interleave
        a, b = first.next, second.next
        first.next, second.next = second, a
        first, second = a, b
    return head

assert to_list(reorder(build([1, 2, 3, 4]))) == [1, 4, 2, 3]
print(to_list(reorder(build([1, 2, 3, 4, 5]))))

[1, 5, 2, 4, 3]


In [ ]:
def largest_rectangle(heights):
    stack, best = [], 0
    for i, h in enumerate(heights + [0]):
        while stack and heights[stack[-1]] >= h:
            height = heights[stack.pop()]
            left = stack[-1] if stack else -1
            best = max(best, height * (i - left - 1))
        stack.append(i)
    return best

def maximal_rectangle(grid):
    heights = [0] * len(grid[0])
    best = 0
    for row in grid:
        heights = [h + 1 if c == "1" else 0 for h, c in zip(heights, row)]
        best = max(best, largest_rectangle(heights))
    return best

assert maximal_rectangle(["0"]) == 0 and maximal_rectangle(["11", "11"]) == 4
print(maximal_rectangle(["10100", "10111", "11111", "10010"]))

6


In [ ]:
from collections import deque
import random

def shortest_subarray(nums, k):
    prefix = [0]
    for x in nums:
        prefix.append(prefix[-1] + x)
    dq, best = deque(), len(nums) + 1
    for i, p in enumerate(prefix):
        while dq and p - prefix[dq[0]] >= k:
            best = min(best, i - dq.popleft())
        while dq and prefix[dq[-1]] >= p:
            dq.pop()
        dq.append(i)
    return best if best <= len(nums) else -1

def brute(nums, k):
    best = len(nums) + 1
    for i in range(len(nums)):
        total = 0
        for j in range(i, len(nums)):
            total += nums[j]
            if total >= k:
                best = min(best, j - i + 1)
    return best if best <= len(nums) else -1

random.seed(4)
for _ in range(300):
    data = [random.randint(-3, 5) for _ in range(random.randint(1, 10))]
    kk = random.randint(1, 8)
    assert shortest_subarray(data, kk) == brute(data, kk)
print(shortest_subarray([2, -1, 2], 3), shortest_subarray([1, 2], 4))

3 -1


## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/dsa/ch05.html